### Importy

In [10]:
import torch
import onnxruntime as ort
import cv2
import numpy as np
from insightface.app import FaceAnalysis

### Sprawdzanie GPU

In [11]:
cuda_available = torch.cuda.is_available()
print(f"PyTorch CUDA: {cuda_available}")
if cuda_available:
    print(f"Urządzenie: {torch.cuda.get_device_name(0)}")
    print(f"Liczba GPU: {torch.cuda.device_count()}")

PyTorch CUDA: True
Urządzenie: NVIDIA GeForce RTX 3060 Laptop GPU
Liczba GPU: 1


In [12]:
providers = ort.get_available_providers()
print(f"Dostępne procesory ONNX: {providers}")

if 'CUDAExecutionProvider' in providers:
    print("InsightFace będzie korzystać z GPU (CUDA).")
else:
    print("Brak CUDAExecutionProvider (CPU).")


Dostępne procesory ONNX: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
InsightFace będzie korzystać z GPU (CUDA).


### Inicjalizacja modelu

In [13]:
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\kubte/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'

### Funkcja do detekcji, alignmentu i ekstrakcji cech

In [14]:
def get_face_embedding(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    faces = app.get(img)
    
    if len(faces) == 0:
        print(f"Undetected face in the image: {image_path}")
        return None

    # Zwracamy embedding pierwszej wykrytej twarzy (zakładamy 1 osobę na foto)
    # faces[0].normed_embedding to wektor 512-wymiarowy
    return faces[0].normed_embedding

In [ ]:
emb1 = get_face_embedding('data/000001.jpg')
emb2 = get_face_embedding('data/000002.jpg')

### Wyliczenie miary podobieństwa (Cosinus similarity)

In [18]:
if emb1 is not None and emb2 is not None:
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    print(f"Podobieństwo: {similarity:.4f}")

Podobieństwo: -0.0046
